# Understanding the **FL mode** in ``armlet``

This tutorial allows users to understand how the **FL mode** of **ARMLET** works (i.e., the default mode that serves to run federated learning experiments).
It corresponds to the function that is called when running the command `armlet exp.mode=federation` (or just `armlet` as it is the default mode).

## Prerequisites (environment configuration and installation)

If you have not configured the environment and installed ``armlet`` yet, you have to:

1. Install [conda](https://www.anaconda.com/docs/getting-started/main) for managing the environments and run the following commands:

```bash
conda create -n armlet python=3.13.5
conda activate armlet
```

2. Install **``armlet``** using `pip`:

```bash
cd ARMLET_DIR
pip install .
```

Then, fill the two following variables to detail the main paths and run the command.

In [1]:
import os

ARMLET_DIR = "../"
OUTPUT_DIR = os.path.join(ARMLET_DIR, "outputs", "tutorial", "FL_mode")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

## Prepare the configuration

Before calling the [run_federation()](https://sara-bouchenak.github.io/ARMLET/api/armlet.run.html#armlet.run.run_federation) main function (in `ARMLET_DIR/armlet/run.py`), **ARMLET** uses the [Hydra](https://hydra.cc/) framework to dynamically load the configurations. For further details about **ARMLET** configuration (such as the configuration groups and values), see [Configuration](https://sara-bouchenak.github.io/ARMLET/user_guide/config/index.html) documentation page.

In this tutorial, we directly prepare the configuration object in Python by creating dictionaries and using [OmegaConf](https://omegaconf.readthedocs.io/en/2.3_branch/), the YAML based hierarchical configuration system used in Hydra.

The first config dictionary to create is the `cfg_paths`, which contains the main paths.

In [2]:
from omegaconf import OmegaConf

cfg_paths = OmegaConf.create({
  "data_dir": os.path.join(ARMLET_DIR, "datasets"),
  "log_dir": os.path.join(ARMLET_DIR, "logs"),
  "output_dir": OUTPUT_DIR,
  "root_dir": ARMLET_DIR,
})

Then, we need to provide in `cfg_data` information related to the data pipeline (dataset, data cleaning methods, train, validation, and test set splitting, distribution for the data partitioning, many other data processing, and data seed).
With the following configurations, we will:
(a) load the Adult dataset;
(b) perform an IID data partitioning between X clients (the number of clients will be detailed after);
(c) split each client dataset into a training set (80% of the client data) and a test set (20%);
(d) concatenate the union of the clients test sets to form the server test set;
(e) clean all data by removing missing values;
and (f) perform standard preprocessing for tabular data.

In [3]:
cfg_data = OmegaConf.create({
    "cleaning": {
        "name": "default",
        "missing_values": {"_target_": "armlet.data.cleaning.missing_values.MissingValuesDataCleaningMethod"},
    },
    "dataset": {
        "_target_": "armlet.data.datasets.load_Adult_dataset",
        "dataset_name": "Adult",
        "path": os.path.join(cfg_paths["data_dir"], "Adult", "raw_data"),
        "sensitive_attributes": ['age', 'gender', 'race'],
    },
    "distribution": {"_target_": "armlet.data.splitter.ArmletDataSplitter.iid"},
    "others": {
        "client_split": 0.2,
        "client_val_split": 0.0,
        "keep_test": False,
        "sampling_perc": 1.0,
        "server_split": 0.0,
        "server_test": False,
        "server_test_union": True,
        "server_val_split": 0.0,
        "uniform_test": False,
    },
    "processing": {
        "one_hot_encoding": {
            "_apply_directly_to_subdata_": False,
            "_target_": "armlet.data.processing.feature_encoding.one_hot_encoding_pipeline",
        },
        "conversion_to_num": {
            "_apply_directly_to_subdata_": True,
            "_target_": "armlet.data.processing.format_conversion.convert_bool_and_cat_to_num",
        },
        "normalization": {
        "_apply_directly_to_subdata_": False,
            "_target_": "armlet.data.processing.normalization.normalization_pipeline",
            "cols_to_exclude": ['age', 'gender', 'race'],
        },
        "conversion_to_tensors": {
            "_apply_directly_to_subdata_": True,
            "_target_": "armlet.data.processing.format_conversion.convert_dataframes_to_tensors",
            "sensitive_attributes": ['age', 'gender', 'race'],
        },
    },
    "seed": 42,
})

`cfg_eval` contains the configuration values related to the evaluation.
In this tutorial, we use the [armlet.eval.evaluators.MultiCriteriaBinaryClassEval](https://sara-bouchenak.github.io/ARMLET/api/armlet.eval.evaluators.html#armlet.eval.evaluators.MultiCriteriaBinaryClassEval) evaluation class and evaluate the global model at each round (on the server side with the server test set).

In [4]:
cfg_eval = OmegaConf.create({
    "_target_": "armlet.eval.evaluators.MultiCriteriaBinaryClassEval",
    "eval_every": 1,
    "locals": False,
    "post_fit": False,
    "pre_fit": False,
    "server": True,
})

`cfg_exp` details information about how to run the experiment.
Here, we choose the federation mode, enable training, specify cpu as the device, and fix an experiment seed.

In [5]:
cfg_exp = OmegaConf.create({
    "device": "cpu",
    "inmemory": True,
    "mode": "federation",
    "seed": 42,
    "train": True,
})

`cfg_logger` is required to specify the logging class to be used during the experiment.
In this tutorial, we use [armlet.utils.log.ArmletLog](https://sara-bouchenak.github.io/ARMLET/api/armlet.utils.log.html#armlet.utils.log.ArmletLog), the minimal logger provided by **ARMLET** (it only saves the results in a JSON file).

In [6]:
cfg_logger = OmegaConf.create({
    "_target_": "armlet.utils.log.ArmletLog",
    "json_log_dir": cfg_paths["output_dir"],
})

`cfg_method` is used to specify the FL algorithm, the ML model, the client hyperparameters, and the server behavior.
Here, we use [armlet.FL_pipeline.FL_algorithms.ArmletCentralized](https://sara-bouchenak.github.io/ARMLET/api/armlet.FL_pipeline.FL_algorithms.html#armlet.FL_pipeline.FL_algorithms.ArmletCentralizedFL), i.e., the standard FedAvg algorithm adapted to **ARMLET** pipeline, and consider a logistic regression model.
Moreover, at each round, each client performs 10 local epochs by using a SGD optimizer with a learning rate of 0.001 and a weight decay of 0.01.

In [7]:
cfg_method = OmegaConf.create({
    "_target_": "armlet.FL_pipeline.FL_algorithms.ArmletCentralizedFL",
    "hyperparameters": {
        "client": {
          "batch_size": 128,
          "local_epochs": 10,
          "loss": {"_target_": "torch.nn.BCELoss"},
          "optimizer": {"lr": 0.001, "name": "SGD", "weight_decay": 0.01},
          "scheduler": {"gamma": 1, "name": "StepLR", "step_size": 1},
        },
        "model": {
          "_target_": "armlet.utils.net.LogRegression",
          "input_size": None, # Automatically adjusted after data loading
          "num_classes": None, # Automatically adjusted after data loading
        },
        "server": {
          "loss": {"_target_": "torch.nn.BCELoss"},
          "time_to_accuracy_target": None,
          "weighted": True,
        },
    },
})

`cfg_protocol` details the protocol of the FL process.
With these configuration values, we run a FL process with 4 clients, for a total of 2 rounds (as a toy example).
During each round, all clients participate to training.

In [8]:
cfg_protocol = OmegaConf.create({
    "eligible_perc": 1.0,
    "n_clients": 4,
    "n_rounds": 2,
})

Finally, we compose each of the previous configuration dictionaries. 

In [9]:
from armlet.utils.configs import ArmletConfiguration

cfg = OmegaConf.create({
    "data": cfg_data,
    "eval": cfg_eval,
    "exp": cfg_exp,
    "logger": cfg_logger,
    "method": cfg_method,
    "paths": cfg_paths,
    "protocol": cfg_protocol,
    "save": {},
})

cfg = ArmletConfiguration(cfg)

Now, we are ready to investigate the `run_federation()` function, which is called when `exp.mode=federation`.

## Run the data pipeline

First, **ARMLET** runs the data pipeline to compute the [DataSplitter](https://makgyver.github.io/fluke/fluke.data.html#fluke.data.DataSplitter), i.e., the Fluke's Python object that contains all data after processing (launching, data cleaning, train, validation, and test sets splitting, data partitioning, normalization, etc.).
As the data pipeline is quite complex, we do not detail it in this tutorial. However, you can find some information about it in the following tutorial: [Understanding the **data pipeline** in `armlet`](https://sara-bouchenak.github.io/ARMLET/getting_started/tutorials/data_pipeline.html).

In [10]:
from armlet.data import data_pipeline

data_splitter, val_data = data_pipeline(cfg)

## Adjust the configurations and save them

Once we have prepared data, we setup the Fluke environment and adjust some configuration values (which are related to data, such as the number of classes and the input size of the ML model).
We can now save the configuration in a YAML file for allowing reproductibility.

In [11]:
from fluke import FlukeENV

FlukeENV().configure(cfg)

In [12]:
input_size = data_splitter.data_container.clients_tr[0].tensors[0].shape[-1]
cfg.method.hyperparameters.model.input_size = input_size
if data_splitter.data_container.num_classes <= 2:
    cfg.method.hyperparameters.model.num_classes = 1  
else:
    cfg.method.hyperparameters.model.num_classes = data_splitter.data_container.num_classes

In [13]:
import yaml

config_path = os.path.join(cfg.paths.output_dir, "config.yaml")
cfg_to_save = cfg.to_dict()
cfg_to_save["paths"]["output_dir"] = "${hydra:runtime.output_dir}"
if "json_log_dir" in cfg_to_save["logger"].keys():
    cfg_to_save["logger"]["json_log_dir"] = "${paths.output_dir}"
yaml.dump(cfg_to_save, open(config_path, "w"))

## Instantiate the FL algorithm, evaluator, and logger

Then, we instantiate the FL algorithm we specified in the `cfg_method` dictionary by using Hydra.
We do the same for the evaluator and the logger with the `cfg_eval` and `cfg_logger` dictionaries, and pass them to the Fluke environment.

In [14]:
import hydra

fl_algo = hydra.utils.instantiate(
    cfg.method,
    n_clients=cfg.protocol.n_clients,
    data_splitter=data_splitter,
    val_data=val_data,
    _convert_="all",
    _recursive_=False,
)

In [15]:
evaluator = hydra.utils.instantiate(
    cfg.eval.exclude("locals", "post_fit", "pre_fit", "server"), 
    n_classes=data_splitter.data_container.num_classes,
    sensitive_attributes=cfg.data.dataset.sensitive_attributes,
)
FlukeENV().set_evaluator(evaluator)

In [16]:
log_name = f"{fl_algo.__class__.__name__} [{fl_algo.id}]"
log = hydra.utils.instantiate(cfg.logger, name=log_name)
log.init(**cfg, exp_id=fl_algo.id)
fl_algo.set_callbacks([log])
FlukeENV().set_logger(log)

╭───────────────────────────────────────── Configuration ──────────────────────────────────────────╮
│ {                                                                                                │
│     'save': {},                                                                                  │
│     'data': {                                                                                    │
│         'cleaning': {                                                                            │
│             'name': 'default',                                                                   │
│             'missing_values': {                                                                  │
│                 '_target_':                                                                      │
│ 'armlet.data.cleaning.missing_values.MissingValuesDataCleaningMethod'                            │
│             }                                                                                    │
│         },                                                                                       │
│         'dataset': {                                                                             │
│             '_target_': 'armlet.data.datasets.load_Adult_dataset',                               │
│             'dataset_name': 'Adult',                                                             │
│             'path': '../datasets/Adult/raw_data',                                                │
│             'sensitive_attributes': [                                                            │
│                 'age',                                                                           │
│                 'gender',                                                                        │
│                 'race'                                                                           │
│             ]                                                                                    │
│         },                                                                                       │
│         'distribution': {                                                                        │
│             '_target_': 'armlet.data.splitter.ArmletDataSplitter.iid'                            │
│         },                                                                                       │
│         'others': {                                                                              │
│             'client_split': 0.2,                                                                 │
│             'client_val_split': 0.0,                                                             │
│             'keep_test': False,                                                                  │
│             'sampling_perc': 1.0,                                                                │
│             'server_split': 0.0,                                                                 │
│             'server_test': False,                                                                │
│             'server_test_union': True,                                                           │
│             'server_val_split': 0.0,                                                             │
│             'uniform_test': False                                                                │
│         },                                                                                       │
│         'processing': {                                                                          │
│             'one_hot_encoding': {                                                                │
│                 '_apply_directly_to_subdata_': False,                                            │
│                 '_target_': 'armlet.data.processing.feature_encoding.one_hot_encoding_pipeline'  │
│             },                                             

## Run the FL experiment

Finally, we call the `run()` function of the FL algorithm object and close the logger once the FL process is completed.
As training progresses, you can follow the evaluation metrics that are computed at the end of each round.
Moreover, by closing `ArmletLog`, all metrics are saved in a JSON file (`results.json`) that can be found in the `OUTPUT_DIR` folder (i.e., `ARMLET_DIR/outputs/tutorial/FL_mode` as defined in the beginning of the tutorial).

In [17]:
try:
    fl_algo.run(cfg.protocol.n_rounds, cfg.protocol.eligible_perc)
except Exception as e:
    log.log(f"Error: {e}")
    FlukeENV().force_close()
    FlukeENV.clear()
    log.close()
    FlukeENV().close_cache()
    raise e
log.close()

[UserWarning] /home/bnaline/anaconda3/envs/armlet/lib/python3.13/site-packages/rich/live.py:260
install "ipywidgets" for Jupyter support

╭──────────────────────────────────────────── Round: 1 ────────────────────────────────────────────╮
│ {                                                                                                │
│     'post-fit': {                                                                                │
│         'training_loss': 0.60185,                                                                │
│         'support': 4,                                                                            │
│         'round': 1                                                                               │
│     },                                                                                           │
│     'global': {                                                                                  │
│         'accuracy': 0.78646,                                                                     │
│         'precision': 0.56932,                                                                    │
│         'recall': 0.56754,                                                                       │
│         'f1': 0.56843,                                                                           │
│         'loss': 0.5378,                                                                          │
│         'round': 1                                                                               │
│     },                                                                                           │
│     'comm_cost': 800                                                                             │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

Memory usage: 938.0 MB [6.06 %]

╭──────────────────────────────────────────── Round: 2 ────────────────────────────────────────────╮
│ {                                                                                                │
│     'post-fit': {                                                                                │
│         'training_loss': 0.49904,                                                                │
│         'support': 4,                                                                            │
│         'round': 2                                                                               │
│     },                                                                                           │
│     'global': {                                                                                  │
│         'accuracy': 0.81407,                                                                     │
│         'precision': 0.65054,                                                                    │
│         'recall': 0.53946,                                                                       │
│         'f1': 0.58981,                                                                           │
│         'loss': 0.4717,                                                                          │
│         'round': 2                                                                               │
│     },                                                                                           │
│     'comm_cost': 800                                                                             │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯

Memory usage: 941.5 MB [6.08 %]

╭────────────────────────────────────── Overall Performance ───────────────────────────────────────╮
│ {                                                                                                │
│     'post-fit': {                                                                                │
│         'training_loss': 0.49904,                                                                │
│         'support': 4,                                                                            │
│         'round': 2                                                                               │
│     },                                                                                           │
│     'global': {                                                                                  │
│         'accuracy': 0.81407,                                                                     │
│         'precision': 0.65054,                                                                    │
│         'recall': 0.53946,                                                                       │
│         'f1': 0.58981,                                                                           │
│         'loss': 0.4717,                                                                          │
│         'round': 2                                                                               │
│     },                                                                                           │
│     'comm_costs': 2000                                                                           │
│ }                                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯